# Phase 1: Topic Clustering
Running unsupervised clustering on pubmed documents.

In [ ]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import json
import os

# Load data
data_path = '../data/raw/pubmed_sma_abstracts.jsonl'
df = pd.read_json(data_path, lines=True)
df = df.dropna(subset=['abstract'])
df = df[df['abstract'].str.strip() != '']
print(f"Loaded {len(df)} abstracts for clustering.")

abstracts = df['abstract'].tolist()

In [ ]:
# Initialize embedding model with a specialized biomedical pre-trained language model
embedding_model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')

# Initialize and fit BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    language="english",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(abstracts)
print("Finished clustering!")
topic_model.get_topic_info().head(10)

In [ ]:
# Visualizations
topic_model.visualize_barchart(top_n_topics=10)

In [ ]:
topic_model.visualize_topics()

In [ ]:
# Save topics back into dataframe
df['topic'] = topics
os.makedirs('../data/processed', exist_ok=True)
df.to_json('../data/processed/clustered_abstracts.jsonl', orient='records', lines=True)
print("Saved clustered data to processed directory.")